# PyPTO Pro 是什么

本节回答四个问题：PyPTO 框架是什么、Pro 范式在其中扮演什么角色、它与 Ascend C 的本质区别、以及它的适用场景。读完本节，你应能向别人解释"为什么存在 PyPTO Pro 这样一个东西"。


## 1. PyPTO：面向 AI 加速器的编程框架

[PyPTO](https://gitcode.com/cann/pypto)（发音：pai p-t-o）是 CANN 推出的一款面向 AI 加速器的高效编程框架，旨在简化算子开发流程，同时保持高性能计算能力，提升算子开发工程师和 Agent 的算子编码和调试调优效率。

对上，它承接 PyTorch 等框架的算子调用；对下，它把开发者编写的程序编译成可在昇腾 NPU 上高效执行的目标代码。PyPTO 面向不同抽象层次的使用者提供了不同的编程入口。


## 2. PyPTO Pro：面向高性能场景的 Tile 级开发范式

**PyPTO Pro 是 PyPTO 面向新一代 Ascend 950PR/950DT 硬件平台的算子高性能场景、自 CANN 9.2.0 版本商用首发的 Pythonic Tile 级开发范式。** 它的三个核心特征：

1. **采用 Python 语法编写 Device 侧 Kernel 核函数**——通过 `@pypto_pro.language.jit()` 装饰器声明 Kernel，用类型标注描述参数，函数体组织数据搬运与计算；
2. **提供 Tensor / Tile / Vector Register 的分层数据表达**——分别对应 GM、片上 Buffer、寄存器三个存储层级（正是上一节的存储地图）；
3. **支持 SIMD 与 SIMT 可组合的混合编程能力**——规整数据用 Tile 批量计算，不规则逻辑用线程逐个处理，两者可在同一 Kernel 中协同。


## 3. Pro 与 Ascend C：编译器 vs 库

Ascend C 是 CANN 原生的 C++ 算子开发语言，同样是"直接控制硬件"的路径。两者的根本区别在于**语义描述能力**：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">PyPTO Pro</th><th style="text-align:left;">Ascend C</th></tr>
<tr><td>语言形态</td><td>Python DSL，Kernel 即 Python 函数</td><td>C++ 库 + 语言扩展</td></tr>
<tr><td>本质</td><td><b>含 IR 语义描述能力，具有编译器属性</b></td><td><b>归根结底是一个库</b></td></tr>
<tr><td>编译行为</td><td>Kernel 函数体被解析为 PyPTO IR，经 IR 转换生成 Device 代码与 Host 侧 Launcher，再编译加载</td><td>Kernel 源码经工具链直接编译，用户视角无 IR/Pass 概念</td></tr>
<tr><td>上手门槛</td><td>Python 基础即可</td><td>需要 C/C++ 与编译链接基础</td></tr>
</table>

这里比较的是开发接口与框架承担的工作。Ascend C 同样依赖编译器工具链；“库式接口”不意味着其工具链没有 IR 或优化 Pass。Pro 将 Python DSL 的解析、IR 变换与 Kernel 编译组织在框架内。

"编译器属性"对你意味着什么？意味着你写的 Python 函数**不是被逐行解释执行**，而是经历一次真正的编译流程。首次启动 Kernel 时依次发生：

1. **绑定** Kernel 实参和启动配置，确定动态参数与编译期特化信息；
2. **解析** Kernel 函数体，生成 PyPTO IR 并进行校验；
3. **转换** IR，生成 Device 代码和 Host 侧 Launcher；
4. **编译加载**产物，然后向指定 Stream 下发 Kernel。

同一 Kernel 对象按**编译签名**区分编译实例——签名相同的重复调用会复用当前进程中的编译结果，这也是"首次调用慢、后续调用快"的原因。第 2 章将详细展开 JIT 编译。


## 4. 适用场景：什么时候用 Pro

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">场景</th><th style="text-align:left;">是否适合</th><th style="text-align:left;">说明</th></tr>
<tr><td>追求极致性能的融合算子</td><td>✅ 核心场景</td><td>Tile 级控制 + Cube/Vector 协同，消除算子间搬运与 Launch 开销</td></tr>
<tr><td>大模型中的热点算子定制</td><td>✅</td><td>如注意力变体、门控/稀疏索引等需要"一处写透"的模块</td></tr>
<tr><td>已有 Python/PyTorch 技术栈的团队</td><td>✅</td><td>无需引入 C++ 工程链路</td></tr>
<tr><td>简单验证算法原型</td><td>⚠️ 可用但非最优</td><td>快速原型同样受益于 Pythonic 表达，但极致调优才是 Pro 的主场</td></tr>
</table>

一句话：**PyPTO Pro 把"性能专家级"的硬件控制能力，装进了 Python 的表达体系里**。代价是你要理解硬件（上一节）、理解编译（本节）、理解并行范式（下一节）——这正是本章剩余部分的路线图。


### 一次调用经历哪些阶段？

阅读 `add_kernel[None,1](x,y,out)` 时，可以沿四个问题跟踪：参数是否符合声明的 shape/dtype；当前编译签名是否已有缓存；若没有，源码如何解析为 IR 并编译加载；设备执行后何时允许 Host 读取结果。

普通 Python 的列表构造、文件读取和 PyTorch Golden 放在 Host。装饰器里的函数虽然使用 Python 语法，却受 DSL 支持范围限制，不能据此认为任意 Python 库都能在设备执行。首次调用的墙钟时间包含编译与加载；之后的调用也包含 Host 下发开销，因此都不能不加区分地称为“纯 Kernel 耗时”。第 2 章讨论签名复用，第 8 章建立测量口径。


## 5. 课后练习

1. （多选）以下哪些是 PyPTO Pro 的核心特征？
   A. 采用 Python 语法编写 Device 侧 Kernel<br/>
   B. 提供 Tensor/Tile/Vector Register 分层数据表达<br/>
   C. 仅支持 SIMD 一种计算方式<br/>
   D. 支持 SIMD 与 SIMT 组合的混合编程
2. （填空）PyPTO Pro 与 Ascend C 的本质区别：Pro 含 ______ 语义描述能力，具有 ______ 属性；而 Ascend C 归根结底是一个 ______。
3. （选择）同一个 Pro Kernel 第二次以相同编译签名调用时更快，主要因为：
   A. Python 解释器做了函数缓存<br/>
   B. 编译结果按签名复用，无需重新编译<br/>
   C. NPU 对重复任务有硬件缓存

In [ ]:
!cat ./answer/01.04_answer.txt

## 6. 本节小结

- PyPTO Pro = 面向 950 高性能场景的 **Pythonic Tile 级开发范式**（CANN 9.2.0 商用首发）；
- 与 Ascend C 的本质区别：**IR 语义描述能力与编译器属性 vs 库**；
- JIT 四步流程：**绑定 → 解析（生成 PyPTO IR）→ 转换（生成 Device 代码与 Launcher）→ 编译加载下发**；
- 主场是追求极致性能的融合算子与热点算子定制。

下一节我们把镜头拉到并行范式：SPMD、SIMD、SIMT 到底是什么关系。